(p3-lab-gradient-descent)=
# P3 실습: 경사하강법과 모델 훈련

**주요 내용**

선형 회귀 모델을 이용하여 경사하강법이 모델의 파라미터를 어떻게 학습하는지 실습한다.

> **손실을 줄이기 위해 모델의 파라미터는 어떻게 반복적으로 업데이트되는가?**

주요 흐름은 다음과 같다.

> **파라미터와 MSE → 배치·스텝·에포크·학습률 → 배치 경사하강법 → 학습률 비교 → 특성 스케일링 → 확률적·미니배치 경사하강법 → `SGDRegressor` → 조기 종료**

로지스틱 회귀와 소프트맥스 회귀는 다루지 않는다.

**환경설정**

In [ ]:
import sys
import sklearn
from packaging.version import Version

assert sys.version_info >= (3, 10)
assert Version(sklearn.__version__) >= Version("1.6.1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rc("font", size=12)
plt.rc("axes", labelsize=13, titlesize=13)
plt.rc("legend", fontsize=11)

## 선형 회귀의 파라미터와 손실

### 실습 데이터

선형 관계를 갖는 간단한 데이터를 만든다.

선형 회귀 모델의 예측식은 다음과 같다.

$$
\hat y = \theta_0 + \theta_1 x
$$

`theta_0`는 편향, `theta_1`은 가중치다.  
훈련의 목표는 데이터에 잘 맞는 두 파라미터를 찾는 것이다.

In [ ]:
rng = np.random.default_rng(42)

X = 2 * rng.random((100, 1))
y = 4 + 3 * X[:, 0] + rng.normal(0, 1, 100)

plt.scatter(X[:, 0], y)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

### MSE를 계산한다

회귀 모델에서는 평균 제곱 오차(MSE)를 손실 함수로 사용할 수 있다.

$$
\mathrm{MSE}(\theta)
=
\frac{1}{m}
\sum_{i=1}^{m}
(\hat y^{(i)} - y^{(i)})^2
$$

손실값이 작을수록 예측값이 타깃에 가깝다.

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

theta = np.array([0.0, 0.0])

X_b = np.c_[np.ones(len(X)), X]
y_pred = X_b @ theta

print("초기 파라미터:", theta)
print("초기 MSE:", mse(y, y_pred))

## 배치, 스텝, 에포크, 학습률

경사하강법의 반복 과정을 이해할 때 다음 네 개념을 구분한다.

- **배치(batch)**: 한 번의 파라미터 업데이트에 사용하는 데이터 묶음
- **스텝(step)**: 한 배치로 예측과 손실을 계산하고 파라미터를 한 번 업데이트하는 과정
- **에포크(epoch)**: 훈련셋 전체를 한 번 사용한 과정
- **학습률(learning rate, $\eta$)**: 한 스텝에서 파라미터를 얼마나 크게 움직일지 정하는 값

배치 크기가 $b$, 훈련셋 크기가 $m$이면 한 에포크의 스텝 수는 대략 $m/b$다.

## 배치 경사하강법

### 파라미터를 손실이 감소하는 방향으로 업데이트한다

배치 경사하강법은 **훈련셋 전체를 하나의 배치**로 사용한다. 따라서 한 에포크에 한 번 파라미터를 업데이트한다.

파라미터 업데이트는 다음 형태로 이루어진다.

$$
\theta^{(\mathrm{new})}
=
\theta^{(\mathrm{old})}
-
\eta
\nabla_\theta \mathrm{MSE}(\theta^{(\mathrm{old})})
$$

실습에서는 MSE의 그레이디언트를 계산하여 이 업데이트를 직접 반복한다.

In [ ]:
def batch_gradient_descent(X_b, y, eta=0.1, n_epochs=100):
    m = len(X_b)
    theta = np.zeros(X_b.shape[1])
    losses = []

    for epoch in range(n_epochs):
        y_pred = X_b @ theta
        error = y_pred - y

        losses.append(np.mean(error ** 2))

        gradients = (2 / m) * X_b.T @ error
        theta = theta - eta * gradients

    return theta, np.array(losses)

theta_bgd, losses_bgd = batch_gradient_descent(
    X_b, y,
    eta=0.1,
    n_epochs=100
)

print("학습된 파라미터:", theta_bgd)
print("마지막 MSE:", losses_bgd[-1])

In [ ]:
plt.plot(losses_bgd)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Batch gradient descent")
plt.show()

### 학습된 직선을 확인한다

경사하강법으로 학습한 파라미터를 이용해 예측선을 그린다.

In [ ]:
x_line = np.array([[0.0], [2.0]])
x_line_b = np.c_[np.ones(len(x_line)), x_line]
y_line = x_line_b @ theta_bgd

plt.scatter(X[:, 0], y)
plt.plot(x_line[:, 0], y_line)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 학습률의 중요성

학습률이 너무 작으면 손실이 천천히 감소하고, 너무 크면 최솟값에 수렴하지 못할 수 있다.

`training_models.ipynb`에서 비교한 세 학습률을 사용해 차이를 확인한다.

- `0.02`: 작은 학습률
- `0.1`: 적절한 학습률의 예
- `0.5`: 큰 학습률의 예

이 값들의 적절성은 데이터와 모델에 따라 달라질 수 있다.

In [ ]:
learning_rates = [0.02, 0.1, 0.5]

lr_results = {}

for eta in learning_rates:
    theta_lr, losses_lr = batch_gradient_descent(
        X_b, y,
        eta=eta,
        n_epochs=100
    )

    lr_results[eta] = {
        "theta": theta_lr,
        "losses": losses_lr
    }

summary = pd.DataFrame({
    eta: {
        "final_MSE": result["losses"][-1],
        "theta_0": result["theta"][0],
        "theta_1": result["theta"][1]
    }
    for eta, result in lr_results.items()
}).T

summary

In [ ]:
for eta in learning_rates:
    plt.plot(
        lr_results[eta]["losses"],
        label=f"eta={eta}"
    )

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.yscale("log")
plt.legend()
plt.title("Learning rate comparison")
plt.show()

### 확인

그래프를 보고 다음을 설명한다.

- 학습률이 작을 때 손실은 어떻게 변하는가?
- 적절한 학습률에서는 어떤 모습인가?
- 학습률이 너무 클 때 어떤 문제가 나타나는가?

## 특성 스케일링과 경사하강법

특성들의 크기가 크게 다르면 경사하강법이 손실의 최솟값을 향해 비효율적으로 이동할 수 있다.

두 특성의 스케일이 매우 다른 데이터를 만들어 비교한다.

In [ ]:
rng = np.random.default_rng(7)

X_scale = np.column_stack([
    rng.uniform(0, 1, 200),
    rng.uniform(0, 1000, 200)
])

y_scale = (
    5
    + 2 * X_scale[:, 0]
    + 0.01 * X_scale[:, 1]
    + rng.normal(0, 0.5, 200)
)

print("첫 번째 특성 범위:",
      X_scale[:, 0].min(), "~", X_scale[:, 0].max())
print("두 번째 특성 범위:",
      X_scale[:, 1].min(), "~", X_scale[:, 1].max())

같은 학습률을 사용해 원자료와 표준화한 데이터에서 배치 경사하강법을 실행한다.

In [ ]:
from sklearn.preprocessing import StandardScaler

def prepare_with_bias(X):
    return np.c_[np.ones(len(X)), X]

X_raw_b = prepare_with_bias(X_scale)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_scale)
X_scaled_b = prepare_with_bias(X_scaled)

_, loss_raw = batch_gradient_descent(
    X_raw_b, y_scale,
    eta=0.1,
    n_epochs=50
)

_, loss_scaled = batch_gradient_descent(
    X_scaled_b, y_scale,
    eta=0.1,
    n_epochs=50
)

In [ ]:
plt.plot(loss_raw, label="Before scaling")
plt.plot(loss_scaled, label="After scaling")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.yscale("log")
plt.legend()
plt.title("Effect of feature scaling")
plt.show()

### 확인

같은 학습률을 사용했는데도 두 학습 과정이 다른 이유를 설명한다.

> **핵심:** 경사하강법에서는 특성 스케일링이 수렴 속도와 안정성에 영향을 줄 수 있다.

## 경사하강법의 세 가지 방식

배치 크기에 따라 경사하강법을 다음처럼 구분할 수 있다.

| 방법 | 한 스텝에 사용하는 데이터 |
|---|---|
| **배치 경사하강법** | 훈련셋 전체 |
| **확률적 경사하강법(SGD)** | 샘플 1개 |
| **미니배치 경사하강법** | 훈련셋의 일부 |

배치 크기가 작을수록 한 에포크 안에서 파라미터가 더 자주 업데이트된다.

### 확률적 경사하강법을 직접 실행한다

한 스텝마다 샘플 하나를 사용하고, 에포크마다 샘플 순서를 섞는다.

In [ ]:
def stochastic_gradient_descent(
    X_b, y,
    eta0=0.05,
    n_epochs=50,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    m = len(X_b)
    theta = np.zeros(X_b.shape[1])
    losses = []
    step = 0

    for epoch in range(n_epochs):
        indices = rng.permutation(m)

        for i in indices:
            xi = X_b[i:i+1]
            yi = y[i:i+1]

            error = xi @ theta - yi
            gradient = 2 * xi.T @ error

            eta = eta0 / (1 + 0.01 * step)
            theta = theta - eta * gradient.ravel()

            step += 1

        losses.append(mse(y, X_b @ theta))

    return theta, np.array(losses)

theta_sgd, losses_sgd = stochastic_gradient_descent(
    X_b, y
)

print("SGD 파라미터:", theta_sgd)
print("마지막 MSE:", losses_sgd[-1])

### 미니배치 경사하강법을 직접 실행한다

한 스텝에서 여러 샘플을 사용한다. 여기서는 배치 크기를 10으로 사용한다.

In [ ]:
def mini_batch_gradient_descent(
    X_b, y,
    batch_size=10,
    eta0=0.1,
    n_epochs=50,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    m = len(X_b)
    theta = np.zeros(X_b.shape[1])
    losses = []
    step = 0

    for epoch in range(n_epochs):
        indices = rng.permutation(m)
        X_shuffled = X_b[indices]
        y_shuffled = y[indices]

        for start in range(0, m, batch_size):
            X_batch = X_shuffled[start:start + batch_size]
            y_batch = y_shuffled[start:start + batch_size]

            error = X_batch @ theta - y_batch
            gradient = (2 / len(X_batch)) * X_batch.T @ error

            eta = eta0 / (1 + 0.01 * step)
            theta = theta - eta * gradient

            step += 1

        losses.append(mse(y, X_b @ theta))

    return theta, np.array(losses)

theta_mbgd, losses_mbgd = mini_batch_gradient_descent(
    X_b, y,
    batch_size=10
)

print("Mini-batch GD 파라미터:", theta_mbgd)
print("마지막 MSE:", losses_mbgd[-1])

### 세 방식을 비교한다

손실 곡선의 모양과 최종 손실을 비교한다.

In [ ]:
theta_bgd_50, losses_bgd_50 = batch_gradient_descent(
    X_b, y,
    eta=0.1,
    n_epochs=50
)

comparison = pd.DataFrame({
    "Batch GD": losses_bgd_50,
    "SGD": losses_sgd,
    "Mini-batch GD": losses_mbgd
})

comparison.tail()

In [ ]:
plt.plot(losses_bgd_50, label="Batch GD")
plt.plot(losses_sgd, label="SGD")
plt.plot(losses_mbgd, label="Mini-batch GD")

plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Gradient descent methods")
plt.show()

### 확인

세 방식의 차이를 **배치 크기와 파라미터 업데이트 횟수**를 중심으로 설명한다.

훈련셋이 100개일 때 한 에포크의 스텝 수는 다음과 같다.

- 배치 경사하강법: 1
- SGD: 100
- 미니배치 경사하강법(`batch_size=10`): 10

## 사이킷런의 `SGDRegressor`

`SGDRegressor`는 확률적 경사하강법을 사용하는 선형 회귀 모델이다.

`training_models.ipynb`에서 다룬 주요 설정 중 다음을 확인한다.

- `max_iter`: 최대 에포크 수
- `tol`: 훈련 중지에 사용하는 허용 오차
- `learning_rate`: 학습률 조정 방식
- `eta0`: 초기 학습률
- `power_t`: `invscaling` 방식의 학습률 감소에 사용
- `n_iter_no_change`: 개선을 기다리는 에포크 수

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline

sgd_reg = make_pipeline(
    StandardScaler(),
    SGDRegressor(
        penalty=None,
        max_iter=1000,
        tol=1e-3,
        learning_rate="invscaling",
        eta0=0.01,
        power_t=0.25,
        n_iter_no_change=5,
        random_state=42
    )
)

sgd_reg.fit(X, y)

print("훈련 MSE:", mse(y, sgd_reg.predict(X)))
print(
    "실제 실행 에포크:",
    sgd_reg.named_steps["sgdregressor"].n_iter_
)

### `LinearRegression`과 비교한다

두 모델이 같은 선형 관계를 학습하는지 파라미터와 MSE를 비교한다.

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X, y)

comparison_models = pd.DataFrame({
    "LinearRegression": [
        lin_reg.intercept_,
        lin_reg.coef_[0],
        mse(y, lin_reg.predict(X))
    ],
    "SGDRegressor": [
        sgd_reg.named_steps["sgdregressor"].intercept_[0]
        - (
            sgd_reg.named_steps["sgdregressor"].coef_[0]
            * sgd_reg.named_steps["standardscaler"].mean_[0]
            / sgd_reg.named_steps["standardscaler"].scale_[0]
        ),
        sgd_reg.named_steps["sgdregressor"].coef_[0]
        / sgd_reg.named_steps["standardscaler"].scale_[0],
        mse(y, sgd_reg.predict(X))
    ]
}, index=["theta_0", "theta_1", "MSE"])

comparison_models

## 조기 종료

경사하강법에서는 최대 에포크까지 항상 훈련할 필요는 없다.

손실이 일정 기간 충분히 개선되지 않으면 훈련을 중단할 수 있다.  
`SGDRegressor`는 `early_stopping=True`로 지정하면 훈련 데이터의 일부를 검증에 사용하여 조기 종료할 수 있다.

In [ ]:
early_stop_reg = make_pipeline(
    StandardScaler(),
    SGDRegressor(
        penalty=None,
        max_iter=1000,
        tol=1e-3,
        learning_rate="invscaling",
        eta0=0.01,
        power_t=0.25,
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=5,
        random_state=42
    )
)

early_stop_reg.fit(X, y)

print(
    "조기 종료 모델의 실제 실행 에포크:",
    early_stop_reg.named_steps["sgdregressor"].n_iter_
)
print(
    "훈련 MSE:",
    mse(y, early_stop_reg.predict(X))
)

## 마무리

이번 실습의 핵심은 다음과 같다.

- 모델 훈련은 손실을 줄이는 파라미터를 찾는 과정이다.
- 경사하강법은 그레이디언트의 반대 방향으로 파라미터를 반복해서 업데이트한다.
- 배치, 스텝, 에포크, 학습률은 서로 다른 개념이다.
- 학습률이 너무 작거나 크면 훈련이 느려지거나 수렴하지 않을 수 있다.
- 특성 스케일링은 경사하강법의 수렴에 영향을 줄 수 있다.
- 배치 크기에 따라 배치 경사하강법, SGD, 미니배치 경사하강법으로 구분할 수 있다.
- `SGDRegressor`는 확률적 경사하강법을 사용하는 선형 회귀 모델이다.
- 필요하면 허용 오차와 조기 종료를 이용해 훈련을 적절한 시점에 멈출 수 있다.